In [1]:
// Tangled @ 2025-07-07T23:19:08-04:00 - DO NOT EDIT!
// Source: /home/brandt/Projects/Repos/umbrella/packages/csp/src/channel.ts

import { channel } from "@thi.ng/csp";

const chan = channel<number>();

(async () => {
	// implicit iterator conversion of the channel
	for await (let x of chan) console.log("received", x);
	console.log("channel closed");
})()

chan.write(1);
chan.write(2);
chan.write(3);
chan.close();

received 1
received 2
received 3
channel closed


In [2]:
// Tangled @ 2025-07-07T22:41:35-04:00 - DO NOT EDIT!
// Source: /home/brandt/Projects/Repos/umbrella/packages/csp/tpl.readme.md

import { channel } from "@thi.ng/csp";

// create CSP channel for bi-directional communication
const chan = channel<number>();

// create first async process (ping)
(async () => {
	while (true) {
		// this op will block until a value becomes available in the channel
		const x = await chan.read();
		// if the channel was closed meanwhile, read() will deliver `undefined`
		if (x === undefined || x > 5) {
			console.log("stopping...");
			// calling close() is idempotent
			// any in-flight writes will still be readable
			chan.close();
			break;
		}
		console.log("ping", x);
		// this op will also block until the other side is reading the value
		await chan.write(x + 1);
	}
	console.log("ping done");
})();

// create second async process (pong, almost identical to ping)
(async () => {
	while (true) {
		// wait until value can be read (or channel closed)
		const x = await chan.read();
		// exit loop if channel closed
		if (x === undefined) break;
		console.log("pong", x);
		// write next value & wait until other side read it
		await chan.write(x + 1);
	}
	console.log("pong done");
})();

// kickoff
chan.write(0);

// ping 0
// pong 1
// ping 2
// pong 3
// ping 4
// pong 5
// stopping...
// ping done
// pong done

ping 0
pong 1
ping 2
pong 3
ping 4
pong 5
stopping...
ping done
pong done


Promise { true }

In [3]:
// Tangled @ 2025-07-07T22:41:35-04:00 - DO NOT EDIT!
// Source: /home/brandt/Projects/Repos/umbrella/packages/csp/tpl.readme.md

import { channel, consumeWith, into, pubsub } from "@thi.ng/csp";

// input channel (optional)
const src = channel<string>({ id: "users" });

// publisher with a topic function
// (topic here is the first character of each received string)
const pub = pubsub<string>(src, (x) => x[0]);

// create topic subscriptions (channel & debug consumer)
// under the hood each topic is a Mult (multiplexed channel)
// subscription channels are automatically named:
// `<src-id>-<topic>-tap<tapid>` (see below)
for (let i of "abc") {
	consumeWith(pub.subscribeTopic(i), (x, ch) => console.log(ch.id, x));
}

// start processing by feeding an iterable of names
await into(src, ["alice", "bert", "bella", "charlie", "arthur"]);

// users-a-tap0 alice
// users-b-tap1 bert
// users-b-tap1 bella
// users-c-tap2 charlie
// users-a-tap0 arthur

// pubsubs & mults are closed recursively once we close the input channel
src.close();

users-a-tap2 alice
users-b-tap3 bert
users-b-tap3 bella
users-c-tap4 charlie
users-a-tap2 arthur
